# Agent Governance in LangGraph with Runtime Policy Controls

This notebook demonstrates a LangGraph financial operations agent that reads customer records, calls an external API, and writes payout output. The goal is to show runtime policy decisions at tool-call boundaries.

Observability systems explain what happened. Runtime policy enforcement explains what was allowed to happen before side effects. Both are useful, and they serve different purposes.


## 1) Prerequisites

Install packages and set credentials.


In [ ]:
# %pip install -q langgraph atensec-thoth requests

import os

if not os.getenv("ATEN_API_KEY"):
    raise RuntimeError("Set ATEN_API_KEY first (https://start.atensecurity.com)")

## 2) Build the agent (financial data workflow)


In [ ]:
from dataclasses import dataclass
import json
import os
from pathlib import Path
from typing import Any, Literal
import uuid

import requests

import thoth
from thoth import ThothPolicyViolation
from thoth.enforcer_client import EnforcerClient
from thoth.models import EnforcementMode, ThothConfig
from thoth.session import SessionContext

from langchain_core.messages import AIMessage, HumanMessage
from langchain_core.tools import tool
from langgraph.graph import END, START, MessagesState, StateGraph
from langgraph.prebuilt import ToolNode


@dataclass(frozen=True)
class Settings:
    api_key: str
    tenant_id: str
    api_url: str
    user_id: str
    workspace_root: Path


def load_settings(workspace_root: str = ".") -> Settings:
    tenant_id = os.getenv("THOTH_TENANT_ID", "basistheory").strip() or "basistheory"
    api_url = os.getenv(
        "THOTH_API_URL", f"https://enforce.{tenant_id}.atensecurity.com"
    ).strip()
    user_id = os.getenv(
        "THOTH_USER_ID", f"{os.getenv('USER', 'operator')}@local"
    ).strip()
    return Settings(
        api_key=os.environ["ATEN_API_KEY"],
        tenant_id=tenant_id,
        api_url=api_url,
        user_id=user_id,
        workspace_root=Path(workspace_root).resolve(),
    )


class FinancialToolchain:
    def __init__(self, workspace_root: Path) -> None:
        self.workspace_root = workspace_root

    def read_customer_records(self, record_count: int) -> str:
        records = [
            {
                "customer_id": f"cust-{idx + 1:03d}",
                "risk_band": "medium" if idx % 2 == 0 else "low",
            }
            for idx in range(record_count)
        ]
        return json.dumps({"record_count": record_count, "records": records}, indent=2)

    def fetch_account_profile(self, url: str) -> str:
        response = requests.get(url, timeout=20)
        response.raise_for_status()
        body = (
            response.json()
            if "application/json" in response.headers.get("content-type", "")
            else {"text": response.text[:3000]}
        )
        return json.dumps(
            {"url": url, "status_code": response.status_code, "body": body}, indent=2
        )

    def write_payout_file(self, path: str, content: str) -> str:
        target = (self.workspace_root / path).resolve()
        if not str(target).startswith(str(self.workspace_root.resolve())):
            raise ValueError(f"path escapes workspace: {path}")
        target.parent.mkdir(parents=True, exist_ok=True)
        target.write_text(content, encoding="utf-8")
        return f"wrote {target}"

    def ask_user_approval(self, prompt: str, action_context: str) -> str:
        return json.dumps(
            {"prompt": prompt, "action_context": action_context}, indent=2
        )


class FinanceState(MessagesState):
    objective: str
    step: int
    force_block: bool
    include_ask_user: bool


def tool_call(name: str, args: dict[str, Any]) -> dict[str, Any]:
    return {
        "id": f"tool-{uuid.uuid4().hex[:10]}",
        "name": name,
        "args": args,
        "type": "tool_call",
    }


def planner(state: FinanceState) -> dict[str, Any]:
    step = int(state.get("step", 0))
    force_block = bool(state.get("force_block", False))
    include_ask_user = bool(state.get("include_ask_user", False))

    if step == 0:
        return {
            "step": 1,
            "messages": [
                AIMessage(
                    content="Read records",
                    tool_calls=[
                        tool_call("read_customer_records", {"record_count": 12})
                    ],
                )
            ],
        }
    if step == 1:
        return {
            "step": 2,
            "messages": [
                AIMessage(
                    content="Fetch profile",
                    tool_calls=[
                        tool_call(
                            "fetch_account_profile",
                            {
                                "url": "https://api.github.com/repos/langchain-ai/langgraph"
                            },
                        )
                    ],
                )
            ],
        }
    if step == 2 and include_ask_user:
        return {
            "step": 3,
            "messages": [
                AIMessage(
                    content="Ask user",
                    tool_calls=[
                        tool_call(
                            "ask_user_approval",
                            {
                                "prompt": "Approve payout?",
                                "action_context": "risky_approval_prompt",
                            },
                        )
                    ],
                )
            ],
        }

    if (step == 2 and not include_ask_user) or (step == 3 and include_ask_user):
        if force_block:
            return {
                "step": 4,
                "messages": [
                    AIMessage(
                        content="Trigger block",
                        tool_calls=[
                            tool_call(
                                "fetch_account_profile", {"url": "https://example.com"}
                            )
                        ],
                    )
                ],
            }
        return {
            "step": 4,
            "messages": [
                AIMessage(
                    content="Write payout",
                    tool_calls=[
                        tool_call(
                            "write_payout_file",
                            {
                                "path": "tmp/payout_plan.csv",
                                "content": "customer_id,action\ncust-001,review\n",
                            },
                        )
                    ],
                )
            ],
        }

    return {
        "messages": [AIMessage(content="Done. Review decisions and audit records.")]
    }


def route(state: FinanceState) -> Literal["tools", "__end__"]:
    last = state["messages"][-1]
    if isinstance(last, AIMessage) and last.tool_calls:
        return "tools"
    return END


def build_tools(governed: FinancialToolchain) -> list[Any]:
    @tool("read_customer_records")
    def read_customer_records(record_count: int) -> str:
        return governed.read_customer_records(record_count)

    @tool("fetch_account_profile")
    def fetch_account_profile(url: str) -> str:
        return governed.fetch_account_profile(url)

    @tool("write_payout_file")
    def write_payout_file(path: str, content: str) -> str:
        return governed.write_payout_file(path, content)

    @tool("ask_user_approval")
    def ask_user_approval(prompt: str, action_context: str) -> str:
        return governed.ask_user_approval(prompt, action_context)

    return [
        read_customer_records,
        fetch_account_profile,
        write_payout_file,
        ask_user_approval,
    ]


def instrument(settings: Settings, enforcement: str) -> FinancialToolchain:
    return thoth.instrument_toolchain(
        FinancialToolchain(settings.workspace_root),
        agent_id="langgraph-finance-agent",
        approved_scope=[
            "read_customer_records",
            "fetch_account_profile",
            "write_payout_file",
            "ask_user_approval",
        ],
        tenant_id=settings.tenant_id,
        user_id=settings.user_id,
        enforcement=enforcement,
        api_key=settings.api_key,
        api_url=settings.api_url,
        purpose="finance-operations",
        data_classification="confidential",
        task_context={
            "initiated_by": settings.user_id,
            "task_id": "cookbook-finance-governance",
        },
    )


def build_graph(tools: list[Any]) -> Any:
    builder = StateGraph(FinanceState)
    builder.add_node("planner", planner)
    builder.add_node("tools", ToolNode(tools, name="tools"))
    builder.add_edge(START, "planner")
    builder.add_conditional_edges("planner", route)
    builder.add_edge("tools", "planner")
    return builder.compile()


def decision_record(decision: Any) -> dict[str, Any]:
    return {
        "decision_envelope_version": decision.decision_envelope_version,
        "enforcement_trace_id": decision.enforcement_trace_id,
        "authorization_decision": decision.authorization_decision
        or decision.decision.value,
        "decision_reason_code": decision.decision_reason_code,
        "action_classification": decision.action_classification,
        "reason": decision.reason,
        "violation_id": decision.violation_id,
        "hold_token": decision.hold_token,
        "risk_score": decision.risk_score,
        "latency_ms": decision.latency_ms,
        "pack_id": decision.pack_id,
        "pack_version": decision.pack_version,
        "rule_version": decision.rule_version,
        "matched_rule_ids": decision.matched_rule_ids,
        "matched_control_ids": decision.matched_control_ids,
        "policy_references": decision.policy_references,
        "model_signals": decision.model_signals,
        "fastml_features": decision.fastml_features,
        "score_components": decision.score_components,
        "top_contributors": decision.top_contributors,
        "decision_evidence": decision.decision_evidence,
        "receipt": decision.receipt,
    }


def violation_record(exc: ThothPolicyViolation) -> dict[str, Any]:
    session = thoth.get_current_session()
    return {
        "event_type": "TOOL_CALL_BLOCK",
        "tool_name": exc.tool_name,
        "reason": exc.reason,
        "violation_id": exc.violation_id,
        "session_id": session.session_id if session else None,
        "metadata": {
            "decision_envelope_version": exc.decision_envelope_version,
            "enforcement_trace_id": exc.enforcement_trace_id,
            "authorization_decision": exc.authorization_decision,
            "decision_reason_code": exc.decision_reason_code,
            "action_classification": exc.action_classification,
            "risk_score": exc.risk_score,
            "latency_ms": exc.latency_ms,
            "pack_id": exc.pack_id,
            "pack_version": exc.pack_version,
            "rule_version": exc.rule_version,
            "matched_rule_ids": exc.matched_rule_ids,
            "matched_control_ids": exc.matched_control_ids,
            "policy_references": exc.policy_references,
            "model_signals": exc.model_signals,
            "fastml_features": exc.fastml_features,
            "score_components": exc.score_components,
            "top_contributors": exc.top_contributors,
            "decision_evidence": exc.decision_evidence,
            "receipt": exc.receipt,
        },
    }


def preview(settings: Settings) -> dict[str, dict[str, Any]]:
    cfg = ThothConfig(
        agent_id="langgraph-finance-agent",
        approved_scope=[
            "read_customer_records",
            "fetch_account_profile",
            "write_payout_file",
            "ask_user_approval",
        ],
        tenant_id=settings.tenant_id,
        user_id=settings.user_id,
        enforcement=EnforcementMode.PROGRESSIVE,
        api_key=settings.api_key,
        api_url=settings.api_url,
    )
    session = SessionContext(cfg)
    client = EnforcerClient(cfg)
    checks = [
        ("read_customer_records", {"record_count": 12}),
        ("write_payout_file", {"path": "tmp/payout_plan.csv"}),
        (
            "fetch_account_profile",
            {"url": "https://example.com", "domain": "example.com"},
        ),
        (
            "ask_user_approval",
            {"prompt": "approve", "action_context": "risky_approval_prompt"},
        ),
    ]
    out = {}
    prior = []
    try:
        for name, args in checks:
            decision = client.check(name, session.session_id, prior + [name], args)
            prior.append(name)
            out[name] = decision_record(decision)
    finally:
        client.close()
    return out


def run(
    settings: Settings, enforcement: str, force_block: bool, include_ask_user: bool
) -> dict[str, Any]:
    graph = build_graph(build_tools(instrument(settings, enforcement)))
    try:
        result = graph.invoke(
            {
                "messages": [
                    HumanMessage(content="Prepare payout recommendations safely.")
                ],
                "objective": "Prepare payout recommendations safely.",
                "step": 0,
                "force_block": force_block,
                "include_ask_user": include_ask_user,
            }
        )
        return {"status": "completed", "final_message": result["messages"][-1].content}
    except ThothPolicyViolation as exc:
        return {"status": "blocked", "audit": violation_record(exc)}


settings = load_settings(".")
settings

## 3) Shadow mode

Run in `observe` mode, then inspect decision payloads that include real schema fields.


In [ ]:
shadow_result = run(
    settings, enforcement="observe", force_block=False, include_ask_user=False
)
shadow_preview = preview(settings)

print("shadow_result=")
print(json.dumps(shadow_result, indent=2))

print("\nshadow_preview=")
print(json.dumps(shadow_preview, indent=2, sort_keys=True))

Audit field names shown above are real SDK keys: `decision_envelope_version`, `enforcement_trace_id`, `authorization_decision`, `decision_reason_code`, `action_classification`, `risk_score`, `matched_rule_ids`, `policy_references`, `decision_evidence`, `receipt`, and related fields.


## 4) Write your first policy

This `.rego` policy implements: BLOCK non-allowlisted outbound domains, STEP_UP file writes, FLAG bulk reads over 10 records, and ALLOW otherwise.


In [ ]:
policy_path = Path("examples/langchain/policies/financial_agent.rego")
policy_path.parent.mkdir(parents=True, exist_ok=True)
policy_text = """package thoth.policies.langgraph.financial_agent

import future.keywords.if
import future.keywords.in

principal := object.get(input, "principal", {})
action := object.get(input, "action", {})
context := object.get(input, "context", {})

action_name := lower(trim(sprintf("%v", [object.get(action, "name", "")]), " 	

"))
action_payload := object.get(action, "payload", {})

destination_domain := lower(trim(sprintf("%v", [object.get(action_payload, "domain", "")]), " 	

"))
record_count := to_number(object.get(action_payload, "record_count", 0))
action_context := lower(trim(sprintf("%v", [object.get(action_payload, "action_context", "")]), " 	

"))

allowlisted_domains := {"api.github.com", "hn.algolia.com", "www.sec.gov"}

deny[reason] if {
  principal != {}
  context != {}
  action_name == "fetch_account_profile"
  destination_domain != ""
  not destination_domain in allowlisted_domains
  reason := sprintf("BLOCK: outbound domain %q is not allowlisted", [destination_domain])
}

deny[reason] if {
  action_name == "ask_user_approval"
  action_context == "risky_approval_prompt"
  reason := "BLOCK: AskUser action_context classified as risky"
}

step_up[reason] if {
  principal != {}
  context != {}
  action_name == "write_payout_file"
  reason := "STEP_UP: payout file writes require approval"
}

flag[reason] if {
  principal != {}
  context != {}
  action_name == "read_customer_records"
  record_count > 10
  reason := sprintf("FLAG: bulk read of %.0f records", [record_count])
}

allow if {
  count(deny) == 0
}
"""
policy_path.write_text(policy_text, encoding="utf-8")
print(policy_path)
print(policy_text)

## 5) Enforce mode

Run enforce-mode checks and then execute a STEP_UP path and a BLOCK path.


In [ ]:
enforce_preview = preview(settings)
print("enforce_preview=")
print(json.dumps(enforce_preview, indent=2, sort_keys=True))

step_up_run = run(
    settings, enforcement="progressive", force_block=False, include_ask_user=False
)
block_run = run(
    settings, enforcement="progressive", force_block=True, include_ask_user=False
)

print("\nstep_up_run=")
print(json.dumps(step_up_run, indent=2, sort_keys=True))

print("\nblock_run=")
print(json.dumps(block_run, indent=2, sort_keys=True))

## 6) AskUser edge case

This path captures an approval request whose `action_context` is risky. In production, this prevents unsafe approval workflows from bypassing governance controls.


In [ ]:
ask_user_run = run(
    settings, enforcement="progressive", force_block=False, include_ask_user=True
)
print(json.dumps(ask_user_run, indent=2, sort_keys=True))

## 7) Production checklist

1. Run shadow mode for at least 7 days per agent profile.
2. Validate decision metadata coverage (`decision_reason_code`, `matched_rule_ids`, `policy_references`).
3. Define explicit approval ownership and SLA for all STEP_UP paths.
4. Verify destination allowlists for every outbound HTTP-capable tool.
5. Test timeout/fail-closed behavior and rollback strategy for policy changes.
6. Add CI checks for agent tool-pattern drift against policy rules.
7. Require policy diff review and controlled promotion from shadow to enforce.
